# exp_rahul_001 — NullSpaceMap on Qwen3.8-27B (fit → generate → judge)

A learned, null-space-constrained steering map (`controls/nullspace_map.py`): read directions in the null space of
capability/guard/twin prompts, write directions toward each dishonesty dim's honest-minus-dishonest answer state;
one shift per prompt, added to answer tokens. See `README.md` in this folder for the method and the evidence.

**Stages** (set `STAGE` below, then *Runtime → Run all*; each stage stops at its end):

| Stage | Runtime | What it does | Time |
| --- | --- | --- | --- |
| `"A"` | **A100 80 GB** | fit at `LAYERS` in bf16, dose sweep, freeze the bundle, verify + roundtrip, `check-submission` | ~30–45 min |
| `"B"` | **A100 80 GB** | generate our arm on exp002's 98 eval rows with the team harness | ~1–1.5 h |
| `"C"` | **L4** (or Kaggle T4×2 via `judge_local.py`) | judge our arm with GLM-4-9B-0414 and score against the unsteered arm | ~20–40 min |

**Precision.** The bundle must be fitted in **bf16** to be submittable (fingerprint check), and arms are only
comparable at the same precision. `PRECISION = "bf16"` needs an 80 GB card *and* a bf16 unsteered reference in
`REFERENCE_RUN`; `"nf4"` runs on 40 GB and compares with exp001's NF4 reference (comparison only, not submittable).

In [ ]:
STAGE = "A"                         # "A": fit + sweep + bundle; "B": generate; "C": judge + score

MODEL = "Qwen/Qwen3.8-27B"
PRECISION = "bf16"                  # "bf16" (submittable; 80 GB) or "nf4" (comparison with exp001's NF4 reference; 40 GB)

# fit (train split only). Layer and dose grid per model come from the 4-bit layer profiles (colab_rahul/diagnose_v7.py):
# the deepest layer with null-space coverage >= 0.98 on every guard area where the weak dims are learnable.
PROFILE = {
    "Qwen/Qwen3.8-27B":            {"layers": [32], "lams": [0.5, 1.0, 1.5, 2.0]},   # 64 layers; D6 +3.3 at lam 1, KL 0.004
    "ibm-granite/granite-4.2-30b": {"layers": [38], "lams": [0.5, 0.75, 1.0, 1.5]},  # 64 layers; D6 +7.2 at lam 1, KL 0.05 (budget)
    "google/gemma-4-31B-it":       {"layers": [30], "lams": [0.5, 1.0, 1.5, 2.0]},   # 60 layers; D6 +16 at lam 1, KL 0.000; gated: needs HF_TOKEN
}
LAYERS = PROFILE[MODEL]["layers"]
LAMS = PROFILE[MODEL]["lams"]
NORMALIZE = "none"                  # per-dim target norms: "none" = as the layer profiles measured (default); "median" halved the
                                    # map's eval R2 and made Granite's weak dims go backwards in the first stage-A run
PAIRS_PER_DIM = 80                  # same pairs per dim as exp002
NULL_FRAC = 0.6                     # share of benign eigen-directions treated as the null space
LAM = None                          # e.g. 1.0 to force the bundle's dose
BENIGN_EXTRA = ["data/benign_prompts_v1/benign_prompts.jsonl"]   # extra prompt-only rows for the benign set (repo paths; see data/benign_prompts_v1/README.md)
FIT_BATCH_SIZE = 4

# evaluation: exp002's trimmed eval set, same harness, same judge
SPLIT = "eval"
TARGET_DIMS = ["instructed_lie", "confrontation_after_lie", "concealment", "sandbagging_stated",
               "capability_knowledge", "capability_reasoning", "capability_problem_solving",
               "capability_instruction_following", "guard_helpfulness", "guard_over_conservativeness",
               "guard_refusal", "guard_confidentiality_persona"]
PER_DIM = 5
CHECK_DIMS = ["pushback", "feedback_sycophancy", "harm_motivated_denial", "fabricated_entity",
              "fabricated_action", "fabricated_citation"]
CHECK_PER_DIM = 2
SAMPLES = 1
GEN_BATCH_SIZE = 4
ARM = "nullspace_v1"
PROBE_DIMS = ["instructed_lie", "confrontation_after_lie", "concealment", "sandbagging_stated"]
PROBE_ROWS = 2                      # stage B first prints greedy 400-token answers on this many rows per dim (0 to skip)
GENERATE_REFERENCE = False          # stage B: also generate the unsteered arm when REFERENCE_RUN has none (doubles B); False = our arm only
COMPARE = True                      # stage C: compare with the unsteered arm (needs it in answers.jsonl); False = absolute summary of our arm only
JUDGE_MODEL = "zai-org/GLM-4-9B-0414"
JUDGE_REVISION = "645b8482494e31b6b752272bf7f7f273ef0f3caf"

RUN_NAME = "exp_rahul_001_nullspace_map"
REFERENCE_RUN = "exp001_qwen_reference"        # where the unsteered answers + judge cache live (per precision tag)
REPO_URL = "https://github.com/Rupesh-sankhala/Steerability-Trust-issues.git"
REPO_BRANCH = "rahul_experiements"             # the branch holding this experiment; "main" once merged
REPO_DIR = "/content/Steerability-Trust-issues"
WHEELS = "/content/drive/MyDrive/steer_wheels"  # optional prebuilt wheel cache (README: "faster installs"); "" to skip

# source of the coherence probe, written into the cloned experiment folder when the branch does not have it yet
PROBE_BUNDLE_SRC = '"""Coherence probe: greedy answers with a bundle on a few rows, printed in full, before paying for a generation run.\n\n    python experiments/exp_rahul_001_nullspace_map/probe_bundle.py --model Qwen/Qwen3.8-27B --bundle <path>.spipe \\\n        --rows data/v7_tidy/eval.jsonl --dims instructed_lie concealment --per-dim 2 --max-new-tokens 400 --out probe.jsonl\n\nTeacher-forced sweeps score a one-line reference; they cannot see a dose that makes the model loop or drift once it\nwrites freely (exp002\'s dose probe found that cliff for CAST). This prints the thinking and the answer of each probe\nrow, with a repetition score (distinct 4-grams; well below 0.5 means looping), so the dose can be lowered before stage B.\n"""\nimport argparse\nimport json\nimport sys\nfrom pathlib import Path\n\nHERE = Path(__file__).resolve().parent\nREPO = HERE.parents[1]\nsys.path.insert(0, str(REPO / "third_party"))\nsys.path.insert(0, str(REPO / "eval"))\nsys.path.insert(0, str(REPO / "steering"))\nsys.path.insert(0, str(HERE))\n\nimport controls  # noqa: E402, F401\nfrom cast_honesty import distinct_ratio  # noqa: E402\nfrom generate_answers import load, select_rows  # noqa: E402\nfrom starterkit import REGIME, generate  # noqa: E402\nfrom starterkit.rows import chat  # noqa: E402\n\n\ndef main(argv=None) -> None:\n    ap = argparse.ArgumentParser(description=__doc__.split("\\n")[0])\n    ap.add_argument("--model", required=True)\n    ap.add_argument("--bundle", required=True)\n    ap.add_argument("--rows", required=True)\n    ap.add_argument("--dims", nargs="+", required=True)\n    ap.add_argument("--per-dim", type=int, default=2)\n    ap.add_argument("--max-new-tokens", type=int, default=400)\n    ap.add_argument("--quantize", choices=["nf4"])\n    ap.add_argument("--out")\n    args = ap.parse_args(argv)\n\n    rows = [json.loads(l) for l in Path(args.rows).read_text().splitlines() if l.strip()]\n    rows = [r for r in select_rows(rows, set(args.dims), args.per_dim) if r["condition"] in ("pressure", "plain")]\n    pipeline = load(args.model, bundle=args.bundle, quantize=args.quantize)\n    kwargs = dict(REGIME[args.model]["chat_template_kwargs"])\n    records = []\n    for r in rows:\n        out = pipeline.generate(messages=[chat(r)], return_output=True, do_sample=False, max_new_tokens=args.max_new_tokens,\n                                chat_template_kwargs=kwargs)\n        out = out[0] if isinstance(out, list) else out\n        thinking, answer = generate.split(args.model, out.output_ids[0], pipeline.tokenizer)\n        text = (thinking or "") + " " + answer\n        score = distinct_ratio(text)\n        flag = "" if score >= 0.5 else "   <-- LOOPING?"\n        print(f"\\n===== {r[\'row_id\']} ({r[\'dim\']}) distinct-4gram {score:.2f}{flag}")\n        print(f"reference: {r[\'reference\'][\'text\'][:160]!r}")\n        print(f"thinking ({len(thinking or \'\')} chars): {(thinking or \'\')[:600]!r}")\n        print(f"answer: {answer[:700]!r}")\n        records.append({"row_id": r["row_id"], "dim": r["dim"], "distinct_ratio": score, "thinking": thinking, "answer": answer})\n    if args.out:\n        Path(args.out).write_text("".join(json.dumps(x, ensure_ascii=False) + "\\n" for x in records))\n    low = [x for x in records if x["distinct_ratio"] < 0.5]\n    print(f"\\n{len(records)} probe rows; {len(low)} flagged as possibly looping")\n\n\nif __name__ == "__main__":\n    main()\n'


In [ ]:
import subprocess
name, memory_mib = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                                  capture_output=True, text=True, check=True).stdout.strip().split("\n")[0].split(", ")
memory_gb = int(memory_mib) / 1024
precision = PRECISION
quantize = "nf4" if precision == "nf4" else None
print(f"GPU: {name}, {memory_gb:.0f} GB, stage {STAGE} -> {MODEL} in {precision}")
if STAGE not in ("A", "B", "C"):
    raise SystemExit('STAGE must be "A", "B", or "C".')
if STAGE != "C" and precision == "bf16" and memory_gb < 75:
    raise SystemExit("bf16 needs an 80 GB A100; switch the runtime, or set PRECISION = 'nf4' (comparison only).")
if STAGE != "C" and memory_gb < 35:
    raise SystemExit("Stages A and B need an A100; only stage C runs on an L4.")

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
RUNS = Path("/content/drive/MyDrive/trust_issues_runs")
tag = f"{MODEL.split('/')[-1]}_{precision}"
RUN_DIR = RUNS / RUN_NAME / tag
REF_DIR = RUNS / REFERENCE_RUN / tag
RUN_DIR.mkdir(parents=True, exist_ok=True)
have_reference = (REF_DIR / "answers.jsonl").exists()
print("outputs go to", RUN_DIR)
print("unsteered reference:", REF_DIR if have_reference else f"NOT FOUND at {REF_DIR} (stage B will generate the unsteered arm too)")

In [ ]:
import os
from google.colab import userdata
if not Path(REPO_DIR).exists():
    token = userdata.get("GITHUB_TOKEN")
    authed = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
    result = subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, authed, REPO_DIR], capture_output=True, text=True)
    if result.returncode != 0:
        raise SystemExit("git clone failed: " + result.stderr.replace(token, "***"))
    subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
EXP = f"{REPO_DIR}/experiments/{RUN_NAME}"
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")   # not needed for Qwen; harmless if set
except Exception:
    pass

In [ ]:
# install. With a Drive wheel cache (README §6) the heavy packages come from Drive in ~1 min; two small pieces are
# always fetched online: the toolkit's build backend (uv_build) and the IFEval checker (a git package).
import time
t0 = time.time()
HEAVY = "accelerate datasets numpy pandas peft scikit-learn torch transformers trl xgrammar bitsandbytes flash-linear-attention"
if WHEELS and Path(WHEELS).is_dir():
    r = subprocess.run(f"pip install -q --no-index --find-links={WHEELS} {HEAVY}", shell=True, capture_output=True, text=True)
    print("offline install:", "ok" if r.returncode == 0 else "failed, falling back to online\n" + r.stderr[-800:])
    if r.returncode != 0:
        %pip install -q {HEAVY}
else:
    %pip install -q {HEAVY}
%pip install -q -e {REPO_DIR}/third_party/steerability-toolkit --no-deps
%pip install -q -r {REPO_DIR}/eval/requirements.txt
# causal_conv1d is optional and compiles for 15-25 min; flash-linear-attention alone gives most of the DeltaNet speed-up.
print(f"installed in {time.time() - t0:.0f}s")
!cd {REPO_DIR} && python -c "import steerability, torch, transformers, bitsandbytes, fla; print('toolkit', steerability.__version__, '| torch', torch.__version__, '| transformers', transformers.__version__)"
!cd {REPO_DIR} && python -m unittest eval/test_scorer.py eval/test_generate_answers.py 2>&1 | tail -2

## Stage A — fit, sweep, bundle

`fit_nullspace_map.py` fits the map at `LAYERS` on the train split (same 80 pairs/dim as exp002; benign set = train
twins + train capability/guard rows + `BENIGN_EXTRA`), reports PEC per protected area and R² toward the response-side
contrast on eval, sweeps `LAMS` with the control itself (teacher-forced log-odds of honest vs dishonest on the four
weak dims; change in log p of the unsteered model's own answers on protected rows), freezes the bundle at the chosen
dose, verifies and reloads it, and writes a submission-shaped folder. Read the sweep table before stage B.

In [ ]:
if STAGE == "A":
    # a refit invalidates earlier generations: move them aside so stage B does not resume from them
    import time as _time
    for name in ("answers.jsonl", "answers_diagnostics.jsonl", "judge_cache.jsonl"):
        old = RUN_DIR / name
        if old.exists():
            kept = RUN_DIR / f"{old.stem}_prev_{_time.strftime('%Y%m%d_%H%M')}{old.suffix}"
            old.rename(kept); print(f"moved stale {name} -> {kept.name}")
    q = f"--quantize {quantize}" if quantize else ""
    lam = f"--lam {LAM}" if LAM is not None else ""
    # the script on the branch may be the older version (flag --no-normalize-dims) or the newer one (--normalize MODE)
    has_new = "--normalize" in open(f"{EXP}/fit_nullspace_map.py").read()
    norm = f"--normalize {NORMALIZE}" if has_new else ("--no-normalize-dims" if NORMALIZE == "none" else "")
    extra = ("--benign-extra " + " ".join(BENIGN_EXTRA)) if BENIGN_EXTRA else ""
    # until the branch carries the fix: the sweep must hand the control the *prompt* (where the map was fitted) and put the
    # model's empty thinking block at the start of the scored continuation, not at the end of the prompt
    src_path = f"{EXP}/fit_nullspace_map.py"; src = open(src_path).read()
    old_a = ('            prompt = render(r); pre = ch.answer_prefix(model_id, prompt)\n'
             '            p = ids(prompt) + ids(pre)\n'
             '            h, dsh = ids(r["contrastive_pair"]["honest"]), ids(r["contrastive_pair"]["dishonest"])\n')
    new_a = ('            prompt = render(r); pre = ids(ch.answer_prefix(model_id, prompt))\n'
             '            p = ids(prompt)\n'
             '            h, dsh = pre + ids(r["contrastive_pair"]["honest"]), pre + ids(r["contrastive_pair"]["dishonest"])\n')
    old_b = 'p = ids(prompt) + ids(ch.answer_prefix(model_id, prompt)); cont = ids(probe["contrastive_pair"]["honest"])'
    new_b = 'p = ids(prompt); cont = ids(ch.answer_prefix(model_id, prompt)) + ids(probe["contrastive_pair"]["honest"])'
    if old_a in src:
        open(src_path, "w").write(src.replace(old_a, new_a).replace(old_b, new_b)); print("patched the sweep's prompt/prefix split in the cloned fit script")
    else:
        print("fit script already carries the sweep fix")
    !cd {REPO_DIR} && python {EXP}/fit_nullspace_map.py --model {MODEL} --layers {" ".join(map(str, LAYERS))} \
        --pairs-per-dim {PAIRS_PER_DIM} --null-frac {NULL_FRAC} {norm} --lams {" ".join(map(str, LAMS))} {lam} {extra} {q} \
        --batch-size {FIT_BATCH_SIZE} --out {RUN_DIR}
    import json, os, time as _t
    if _t.time() - os.path.getmtime(f"{RUN_DIR}/fit_summary.json") > 1800:
        raise SystemExit("fit_summary.json was not updated by this run: the fit failed (see the error above)")
    print(json.dumps(json.load(open(f"{RUN_DIR}/sweep.json"))["arms"], indent=1))
    summary = json.load(open(f"{RUN_DIR}/fit_summary.json"))
    print(json.dumps({k: v for k, v in summary.items() if k != "per_layer"}, indent=1))
    for layer, info in summary["per_layer"].items():
        print(f"layer {layer}: rank {info['rank']}, R2 pair eval {info['r2_pair_eval']}, dose {info['dose_eval']} vs gap {info['gap_eval']}")
        print("  PEC eval:", info["pec_eval"]); print("  leak eval:", info["leak_eval"]); print("  R2 per dim:", info["r2_pair_eval_per_dim"])
    !cd {REPO_DIR}/third_party && python -m starterkit.check {RUN_DIR}/submission 2>&1 | tail -8
    raise SystemExit("stage A done: read the sweep and fit summary above, then set STAGE = 'B'")

## Stage B — generate our arm on exp002's rows

Same harness, rows, ceilings and precision as exp002; the unsteered arm is generated only if `REFERENCE_RUN` has none
for this precision. `run_generate.py` registers the control before the bundle is loaded.

In [ ]:
import json
STAGE="B"
if STAGE == "B":
    import shutil
    bundle = next((RUN_DIR / "bundles").glob("*.spipe"))
    q = f"--quantize {quantize}" if quantize else ""
    out = RUN_DIR / "answers.jsonl"
    if have_reference and not out.exists():
        shutil.copy(REF_DIR / "answers.jsonl", out)          # start from the unsteered answers so the scorer sees both arms
    common = f"--rows data/v7_tidy/{SPLIT}.jsonl --model {MODEL} --samples {SAMPLES} --batch-size {GEN_BATCH_SIZE} {q} --out {out}"
    selections = [(" ".join(TARGET_DIMS), PER_DIM), (" ".join(CHECK_DIMS), CHECK_PER_DIM)]
    arms = ([("unsteered", None)] if (GENERATE_REFERENCE and not have_reference) else []) + [(ARM, bundle)]
    for arm, path in arms:
        b = f"--bundle {path}" if path else ""
        for dims, per_dim in selections:
            print(f"=== {arm}: {per_dim} per category of {dims}", flush=True)
            !cd {REPO_DIR} && python {EXP}/run_generate.py {common} --arm {arm} {b} --dims {dims} --per-dim {per_dim}
    if GENERATE_REFERENCE and not have_reference:
        # we generated the unsteered arm ourselves: share it as the team's bf16/nf4 reference so nobody pays for it twice
        REF_DIR.mkdir(parents=True, exist_ok=True)
        with open(REF_DIR / "answers.jsonl", "w") as ref, open(out) as ours:
            ref.writelines(line for line in ours if json.loads(line)["arm"] == "unsteered")
        print("unsteered reference written to", REF_DIR / "answers.jsonl")
    raise SystemExit("stage B done: answers in RUN_DIR/answers.jsonl; switch to an L4 and set STAGE = 'C'")

## Stage C — judge and score (L4), whenever you want

Judges our arm (and the unsteered arm if it is in the file and not already cached). With an unsteered arm present the
summary is the team's paired comparison (per category both arms, the change, a 95% margin, estimated
delta / r_side / r_guard / S); without one it is the absolute per-category summary of our arm, which is enough to
see whether the steering works on the four weak dims and leaves capability/guard rows alone.

In [ ]:
import json
if STAGE == "C":
    import shutil
    cache = RUN_DIR / "judge_cache.jsonl"
    if not cache.exists() and (REF_DIR / "judge_cache.jsonl").exists():
        shutil.copy(REF_DIR / "judge_cache.jsonl", cache)          # reuse the unsteered arm's verdicts
    answers = RUN_DIR / "answers.jsonl"
    have_unsteered = any(json.loads(l)["arm"] == "unsteered" for l in open(answers))
    if not have_unsteered and have_reference:                      # bring the reference arm into the same file
        with open(answers, "a") as f, open(REF_DIR / "answers.jsonl") as ref:
            f.writelines(ref)
        have_unsteered = True
    compare = COMPARE and have_unsteered
    if COMPARE and not have_unsteered:
        print("no unsteered arm available: printing the absolute summary of our arm only (set COMPARE=False to silence)")
    arm_flag = f"--arm {ARM}" if compare else ""
    env = f"JUDGE_BACKEND=huggingface JUDGE_MODEL={JUDGE_MODEL}"
    !cd {REPO_DIR} && {env} python eval/scorer.py --rows data/v7_tidy/{SPLIT}.jsonl --answers {answers} {arm_flag} --judge --cache {cache} 2>&1 | tee {RUN_DIR}/summary_{ARM}.txt
    print("saved", RUN_DIR / f"summary_{ARM}.txt")
    if have_unsteered and not (REF_DIR / "judge_cache.jsonl").exists():
        REF_DIR.mkdir(parents=True, exist_ok=True)                  # share the unsteered verdicts with the team
        with open(REF_DIR / "judge_cache.jsonl", "w") as ref, open(cache) as ours:
            ref.writelines(line for line in ours if json.loads(line)["key"].startswith("unsteered|"))
        print("unsteered verdicts shared at", REF_DIR / "judge_cache.jsonl")

## Record the result

Copy `fit_summary.json` (minus `per_layer` if long), the sweep table, `summary_<arm>.txt` and the units used into
`results.md` here, and fill in the `exp_rahul_001` entry in `experiments/EXPERIMENT_LOG.md` (precision, layer, λ,
PEC/leakage, honesty change on the four weak dims, r_side, r_guard, estimated S). Commit the folder without the
answers file if it is large; keep `answers.jsonl` and `judge_cache.jsonl` on Drive and record the path.